In [2]:

from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.cluster import DBSCAN

from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy

from sklearn.decomposition import TruncatedSVD
import numpy as np
import seaborn as sns

In [4]:
processed_dir = Path("../data/processed")
results_dir = Path("../results/clustering")
model_dir = Path("../models")

matrice_tfidf_bigrammes = load_npz(processed_dir / "tfidf_matrix_bigrammes.npz")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
data = pd.read_csv(processed_dir / "sample_clean.csv")

assert matrice_tfidf_bigrammes.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(data["id"])

In [8]:
df = data.copy()

In [12]:
df

,id,products,advantages,description_texte,details,fabric,weight,pays
0,1,Active classic boxers,There's a reason why our boxers are a cult fav...,active classic boxers - there s a reason why o...,"silky capilene 1 fabric is ultralight, breatha...",3.7-oz 100% all-recycled polyester with gladio...,99 g 3.5 oz,Mexico
1,2,Active sport boxer briefs,Skinning up Glory requires enough movement wit...,active sport boxer briefs - skinning up glory ...,"stretch mesh provides support, open-weave mesh...","4.6-oz 93% polyester 71% recycled /7% spandex,...",60 g 2.1 oz,Israel
2,3,Active sport briefs,These superbreathable no-fly briefs are the mi...,active sport briefs - these superbreathable no...,"stretch mesh provides support, open-weave mesh...","4.6-oz 93% polyester 71% recycled /7% spandex,...",49 g 1.7 oz,Israel
3,4,Alpine guide pants,"Skin in, climb ice, switch to rock, traverse a...","alpine guide pants - skin in, climb ice, switc...",durable stretch-woven polyester 47% recycled w...,7.6-oz 90-denier 92% polyester 47% recycled /8...,597 g 20.7 oz,Vietnam
4,5,Alpine wind jkt,"On high ridges, steep ice and anything alpine,...","alpine wind jkt - on high ridges, steep ice an...","lightweight, breathable polyester ripstop fabr...",shell 1.3-oz 20-denier 100% polyester ripstop....,331 g 11.5 oz,China
...,...,...,...,...,...,...,...,...
495,496,Cap 2 bottoms,Cut loose from the maddening crowds and search...,cap 2 bottoms - cut loose from the maddening c...,capilene 2 fabric has an open knit that invite...,4-oz 100% polyester 54% recycled with gladiodo...,155 g 5.5 oz,Mexico
496,497,Cap 2 crew,This crew takes the edge off fickle weather. I...,cap 2 crew - this crew takes the edge off fick...,capilene 2 fabric has an open knit that invite...,4-oz 100% polyester 54% recycled with gladiodo...,170 g 6 oz,Mexico
497,498,All-time shell,No need to use that morning Times as an umbrel...,all-time shell - no need to use that morning t...,2.5-layer 100% all-recycled polyester shell wi...,"shell 2.5-layer, 4.1-oz 50-denier 100% all-rec...",493 g 17.4 oz,China
498,499,All-wear cargo shorts,All-Wear Cargo Shorts bask in the glory of swe...,all-wear cargo shorts - all-wear cargo shorts ...,lightweight organic cotton canvas zip fly with...,5.2-oz 100% organic cotton canvas. recyclable ...,349 g 12.1 oz,Thailand


In [5]:
X = matrice_tfidf_bigrammes
dense = X.toarray() # avoid risk to obtain only sparse matrix
dense

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

# apply truncated SVD

In [22]:
svd_model = TruncatedSVD(n_components=10, n_iter=100, random_state=122) # try extract 10 topics
lsa = svd_model.fit_transform(X)
lsa # U matrix

array([[ 0.23009936,  0.15266192,  0.1791427 , ...,  0.01696084,
        -0.06909071, -0.09189423],
       [ 0.2088842 ,  0.12989294,  0.07190223, ...,  0.01404025,
         0.07245042, -0.132492  ],
       [ 0.21035297,  0.14328079,  0.07325795, ...,  0.05416186,
         0.12614319, -0.10848009],
       ...,
       [ 0.23526127,  0.25803631, -0.10526834, ..., -0.17855235,
        -0.04987556,  0.06400073],
       [ 0.30324009,  0.05937415, -0.08282045, ...,  0.04026801,
        -0.25303146, -0.03417576],
       [ 0.33123492,  0.07792216, -0.11448919, ...,  0.03010939,
        -0.22427108, -0.01823285]])

Plus la valeur absolue est importante plus le document est important pour le topic. On peut filtrer les valeurs négatives.

In [10]:
topic_encoded_df = pd.DataFrame(lsa, columns = [f"topic_{i+1}"for i in range(10)], index = df.index)
topic_encoded_df

,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10
0,0.230099,0.152662,0.179143,0.066454,-0.157534,0.016858,0.036626,0.016961,-0.069091,-0.091894
1,0.208884,0.129893,0.071902,0.058431,-0.044517,0.007675,0.087649,0.014040,0.072450,-0.132492
2,0.210353,0.143281,0.073258,0.011576,-0.048827,0.003804,0.029522,0.054162,0.126143,-0.108480
3,0.221259,0.232780,-0.106838,-0.101653,0.018347,-0.006143,-0.003793,-0.052910,-0.021439,-0.020934
4,0.222464,0.255642,-0.091549,-0.180155,-0.018836,-0.009032,-0.050497,-0.016784,0.026773,0.043290
...,...,...,...,...,...,...,...,...,...,...
495,0.227704,0.174281,0.324097,0.021265,-0.337688,0.002797,-0.064852,0.041596,-0.223260,-0.261009
496,0.243366,0.183386,0.389978,-0.035912,-0.386307,-0.002649,-0.125987,0.100833,-0.208491,-0.233271
497,0.235261,0.258036,-0.105268,-0.204723,0.002735,-0.031853,-0.208136,-0.178552,-0.049876,0.064001
498,0.303240,0.059374,-0.082820,0.369229,0.116403,0.001658,0.023478,0.040268,-0.253031,-0.034176


In [13]:
topic_encoded_df["keywords"] = df['products']
topic_encoded_df

,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,keywords
0,0.230099,0.152662,0.179143,0.066454,-0.157534,0.016858,0.036626,0.016961,-0.069091,-0.091894,Active classic boxers
1,0.208884,0.129893,0.071902,0.058431,-0.044517,0.007675,0.087649,0.014040,0.072450,-0.132492,Active sport boxer briefs
2,0.210353,0.143281,0.073258,0.011576,-0.048827,0.003804,0.029522,0.054162,0.126143,-0.108480,Active sport briefs
3,0.221259,0.232780,-0.106838,-0.101653,0.018347,-0.006143,-0.003793,-0.052910,-0.021439,-0.020934,Alpine guide pants
4,0.222464,0.255642,-0.091549,-0.180155,-0.018836,-0.009032,-0.050497,-0.016784,0.026773,0.043290,Alpine wind jkt
...,...,...,...,...,...,...,...,...,...,...,...
495,0.227704,0.174281,0.324097,0.021265,-0.337688,0.002797,-0.064852,0.041596,-0.223260,-0.261009,Cap 2 bottoms
496,0.243366,0.183386,0.389978,-0.035912,-0.386307,-0.002649,-0.125987,0.100833,-0.208491,-0.233271,Cap 2 crew
497,0.235261,0.258036,-0.105268,-0.204723,0.002735,-0.031853,-0.208136,-0.178552,-0.049876,0.064001,All-time shell
498,0.303240,0.059374,-0.082820,0.369229,0.116403,0.001658,0.023478,0.040268,-0.253031,-0.034176,All-wear cargo shorts


In [14]:
# Assign each document to the topic it is the most linked to
topic_encoded_df["class_pred"] = [np.argmax(doc) for doc in lsa]
topic_encoded_df["class_pred"].value_counts()

class_pred
0    219
1     76
2     54
8     51
7     31
3     28
6     21
9     12
5      8
Name: count, dtype: int64

Le topic 1 récupère beaucoup trop de valeurs par rapport aux autres.

In [17]:
topic_encoded_df.head(1)

,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,keywords,class_pred
0,0.230099,0.152662,0.179143,0.066454,-0.157534,0.016858,0.036626,0.016961,-0.069091,-0.091894,Active classic boxers,0


In [16]:
topic_encoded_df.dtypes

topic_1       float64
topic_2       float64
topic_3       float64
topic_4       float64
topic_5       float64
topic_6       float64
topic_7       float64
topic_8       float64
topic_9       float64
topic_10      float64
keywords       object
class_pred      int64
dtype: object

# print the confusion matrix for the topic against the target variable

On représente les topics dans un scatterplot. Comment représenter 10 topics ? Plus un point est proche d'un axe plus il est proche du topic (métrique distance cosinus)

In [ ]:
# cols = ['target', 'class_pred']
# topic_encoded_df[cols] = topic_encoded_df[cols].astype(str)

# évaluer le truncateSVD

Pour générer la heatmap, nous avons besoin de class_pred

In [ ]:
# # Create heatmap with plotly
# import plotly.figure_factory as ff
# from sklearn.metrics import confusion_matrix

# cm = confusion_matrix(y_true = topic_encoded_df["target"], y_pred = topic_encoded_df["class_pred"]) / len(topic_encoded_df)

In [36]:
cm

array([[0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   ],
       [0.

In [ ]:
# fig = ff.create_annotated_heatmap(cm.round(2), x = [i for i in range(20)], y = [i for i in range(20)])
# fig.update_layout(width = 1200)

# fig.show()